# Module 04 — System Prompts & Agent Identity

The **system prompt** is the agent's instruction manual. It defines:
- Who the agent is (persona, name, tone)
- What it should and shouldn't do
- How it should format responses
- Any domain-specific context it needs

A well-crafted system prompt is often the difference between a generic chatbot and a focused, useful agent.

In [6]:
from strands import Agent

# Without a system prompt — generic behavior
generic_agent = Agent(model="us.anthropic.claude-haiku-4-5-20251001-v1:0")
print("Generic agent:")
generic_agent("Tell me about Puttu & Kadala Curry")

Generic agent:


ResourceNotFoundException: An error occurred (ResourceNotFoundException) when calling the ConverseStream operation: Model use case details have not been submitted for this account. Fill out the Anthropic use case details form before using the model. If you have already filled out the form, try again in 15 minutes.

In [11]:
# With a focused system prompt — specialized behavior
chef_agent = Agent(
    model="us.anthropic.claude-haiku-4-5-20251001-v1:0",
    system_prompt="""You are Chef Subbanna, an enthusiastic South Indian chef with 30 years of experience.
You speak with warmth and passion about food. You always:
- Give practical cooking tips
- Mention the importance of fresh, quality ingredients
- Keep responses concise (3-5 sentences max)
- End with an encouraging phrase like 'Buon appetito!'
"""
)

print("Chef Subbanna:")
chef_agent("Tell me about Kadala Curry.")

Chef Subbanna:


ResourceNotFoundException: An error occurred (ResourceNotFoundException) when calling the ConverseStream operation: Model use case details have not been submitted for this account. Fill out the Anthropic use case details form before using the model. If you have already filled out the form, try again in 15 minutes.

Same model, completely different personality and focus.

---

### Dynamic system prompts

You can inject runtime data into the system prompt — like the current date, user info, or environment config.

In [16]:
from datetime import datetime

def create_agent_for_user(username: str, dietary_restrictions: list[str]):
    """Factory function that creates a personalized agent."""
    
    restrictions_text = ", ".join(dietary_restrictions) if dietary_restrictions else "none"
    today = datetime.today().strftime("%Y-%m-%d")
    
    system_prompt = f"""You are a personal food assistant.
Today's date: {today}
User: {username}
Dietary restrictions: {restrictions_text}

Always respect the user's dietary restrictions in every recommendation.
Be friendly and concise.
"""
    
    return Agent(
        model="us.anthropic.claude-haiku-4-5-20251001-v1:0",
        system_prompt=system_prompt
    )


# Create a personalized agent
my_agent = create_agent_for_user(
    username="Yeshwanth",
    dietary_restrictions=["vegan", "gluten-free"]
)

my_agent("What should I have for lunch today?")

ResourceNotFoundException: An error occurred (ResourceNotFoundException) when calling the ConverseStream operation: Model use case details have not been submitted for this account. Fill out the Anthropic use case details form before using the model. If you have already filled out the form, try again in 15 minutes.

### Naming your agent

The `name` parameter is useful for logging, tracing, and multi-agent systems.

In [19]:
agent = Agent(
    name="NutriBot",
    model="us.anthropic.claude-haiku-4-5-20251001-v1:0",
    system_prompt="You are NutriBot, a nutrition expert. Give evidence-based dietary advice in 2-3 sentences."
)

print(f"Agent name: {agent.name}")
agent("Is coffee good or bad for you?")

Agent name: NutriBot


ResourceNotFoundException: An error occurred (ResourceNotFoundException) when calling the ConverseStream operation: Model use case details have not been submitted for this account. Fill out the Anthropic use case details form before using the model. If you have already filled out the form, try again in 15 minutes.

### Modifying the system prompt after creation

You can update `agent.system_prompt` at any time — useful for injecting retrieved context (like memory) before a conversation.

In [6]:
agent = Agent(
    model="us.anthropic.claude-haiku-4-5-20251001-v1:0",
    system_prompt="You are a helpful food assistant."
)

# Simulate injecting retrieved user preferences
retrieved_preferences = "- Loves spicy food\n- Vegetarian\n- Allergic to peanuts"
agent.system_prompt += f"\n\n## Known user preferences:\n{retrieved_preferences}"

print("Updated system prompt:")
print(agent.system_prompt)
print()

agent("What should I order at an Indian restaurant?")

Updated system prompt:
You are a helpful food assistant.

## Known user preferences:
- Loves spicy food
- Vegetarian
- Allergic to peanuts

# Indian Restaurant Recommendations

Based on your preferences, here are some great options:

## Main Dishes
- **Chana Masala** - Spiced chickpeas in a tangy tomato sauce (vegetarian & can be made quite spicy!)
- **Baingan Bharta** - Roasted eggplant curry with aromatic spices
- **Paneer Tikka Masala** - Creamy tomato-based sauce with cottage cheese (you can request extra spice)
- **Saag Paneer** - Spinach and cheese curry with warming spices

## Sides
- **Naan or Roti** - To scoop up the curries
- **Aloo Gobi** - Potato and cauliflower with turmeric and cumin
- **Dal Makhani** - Lentil curry (rich and flavorful)

## Tips
✓ **Ask for spice level** - Request "extra spicy" or "vindaloo" level heat
✓ **Double-check dishes** - Confirm no peanuts are used (they're sometimes in gravies or sprinkled on top)
✓ **Mention allergies** - Always inform your ser

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': '# Indian Restaurant Recommendations\n\nBased on your preferences, here are some great options:\n\n## Main Dishes\n- **Chana Masala** - Spiced chickpeas in a tangy tomato sauce (vegetarian & can be made quite spicy!)\n- **Baingan Bharta** - Roasted eggplant curry with aromatic spices\n- **Paneer Tikka Masala** - Creamy tomato-based sauce with cottage cheese (you can request extra spice)\n- **Saag Paneer** - Spinach and cheese curry with warming spices\n\n## Sides\n- **Naan or Roti** - To scoop up the curries\n- **Aloo Gobi** - Potato and cauliflower with turmeric and cumin\n- **Dal Makhani** - Lentil curry (rich and flavorful)\n\n## Tips\n✓ **Ask for spice level** - Request "extra spicy" or "vindaloo" level heat\n✓ **Double-check dishes** - Confirm no peanuts are used (they\'re sometimes in gravies or sprinkled on top)\n✓ **Mention allergies** - Always inform your server about your peanut allergy\n\n

---

### Key takeaways

- `system_prompt` shapes the agent's persona, constraints, and behavior
- Use f-strings to inject dynamic context (date, user info, retrieved data)
- `name` helps with logging and multi-agent orchestration
- You can update `agent.system_prompt` at runtime — this is how memory hooks inject context

Next up: **04_hooks.ipynb** — reacting to agent lifecycle events.